# SFH14 Forecast Test

This notebook is a lightweight diagnostic entrypoint for the current forecast stack.
It keeps the current managed forecast model unchanged and focuses on inspecting SFH14.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")

profiles_module = importlib.import_module("configs.profiles")
forecast_diagnostics_module = importlib.import_module("scripts.utils.forecast_diagnostics")
importlib.reload(profiles_module)
importlib.reload(forecast_diagnostics_module)

compose_experiment_config = profiles_module.compose_experiment_config
collect_load_step1_diagnostics = forecast_diagnostics_module.collect_load_step1_diagnostics
apply_heatpump_jump_relief = forecast_diagnostics_module.apply_heatpump_jump_relief


In [ ]:
device_request = None
require_cuda = False

cfg = compose_experiment_config()
cfg.data.agent_profiles = ["SFH12", "SFH14", "SFH16", "SFH18", "SFH20"]
cfg.env.num_agents = len(cfg.data.agent_profiles)
cfg.forecast.heatpump_jump_relief_enabled = False
cfg.forecast.heatpump_jump_relief_threshold_kw = 0.8
cfg.forecast.heatpump_jump_relief_min_weight = 0.3

focus_profile = "SFH14"
single_day_total_date = "2020-01-04"
single_day_component_focus_start = "2020-01-04 10:00"
single_day_component_focus_end = "2020-01-04 12:30"
single_day_component_weights = (0.0, 0.05, 0.10, 0.20, 0.30, 1.0)

summary = {
    "project_root": str(project_root),
    "focus_profile": focus_profile,
    "single_day_total_date": single_day_total_date,
    "single_day_component_focus_start": single_day_component_focus_start,
    "single_day_component_focus_end": single_day_component_focus_end,
    "single_day_component_weights": single_day_component_weights,
}
summary


## Notes

The current working hypothesis is that SFH14 local total-load spikes come from a combined effect:
`household spike miss + heatpump conservative lag`.

This notebook intentionally keeps the managed forecast model untouched and is meant for diagnosis only.


In [ ]:
# Optional starting point for deeper analysis when managed artifacts are available.
# diagnostics = collect_load_step1_diagnostics(cfg)
# diagnostics["step1_metrics"].head()
